# GDV Final EDA – Football Shot Analysis

**Project topic:** Football event data visualization  
**Dataset:** StatsBomb Open Data, FIFA World Cup 2022, Serbia vs Switzerland  
**Goal of this notebook:** Build the static visual analysis required for the *Fundamentals of Data Visualization (GDV)* part of the project.

This notebook focuses on the GDV requirements:

- dataset description and motivation
- preprocessing and relevant variables
- exploratory visualizations
- clear visual encodings
- short interpretation of each figure
- figures that can later be used in the report and pitch slides

The interactive dashboard for IVI is handled separately in `dashboard/app.py`.


## 1. Setup

We use `pandas` for data handling, `matplotlib` and `seaborn` for static visualizations, and `statsbombpy` for loading open football event data.
The notebook is written so it can be run from the repository root or from the `notebooks/` folder.


In [ ]:
from pathlib import Path
import ast
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from statsbombpy import sb
    STATSBOMB_AVAILABLE = True
except ImportError:
    STATSBOMB_AVAILABLE = False

warnings.filterwarnings("ignore")

# Make plots readable and consistent
sns.set_theme(style="whitegrid", context="notebook")

# Robust project paths
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, DATA_DIR, FIGURE_DIR


## 2. Load data

The analysis focuses on the match **Serbia vs Switzerland** from the FIFA World Cup 2022.
In StatsBomb Open Data this match has the match id `3857256`.

The notebook first tries to load the event data directly from StatsBomb.
If this is not possible, it falls back to the existing local CSV file `data/raw/shots_single_match.csv`.


In [ ]:
MATCH_ID = 3857256
COMPETITION_ID = 43
SEASON_ID = 106

def load_shots():
    """Load shot data from StatsBomb Open Data or from local CSV fallback."""
    local_file = DATA_DIR / "shots_single_match.csv"

    if STATSBOMB_AVAILABLE:
        try:
            events = sb.events(match_id=MATCH_ID)
            shots = events[events["type"] == "Shot"].copy()
            print(f"Loaded {len(shots)} shots from StatsBomb Open Data.")
            return shots
        except Exception as exc:
            print("Could not load from StatsBomb. Falling back to local CSV.")
            print(exc)

    if local_file.exists():
        shots = pd.read_csv(local_file)
        print(f"Loaded {len(shots)} shots from local CSV: {local_file}")
        return shots

    raise FileNotFoundError(
        "No StatsBomb access and no local fallback file found. "
        "Expected data/raw/shots_single_match.csv"
    )

shots_raw = load_shots()
shots_raw.head()


## 3. Preprocessing

The raw event data contains nested fields and many columns.
For the final visual analysis we only keep variables that are relevant for the research questions.

Relevant variables:

- `team`
- `player`
- `minute`
- `period`
- `location`
- `shot_outcome`
- `shot_statsbomb_xg`
- `shot_body_part`
- `shot_type`
- `play_pattern`

We also derive:

- `x` and `y` from the shot location
- `distance_to_goal`
- `is_goal`


In [ ]:
def parse_location(value):
    """Parse StatsBomb location field safely."""
    if isinstance(value, list) and len(value) >= 2:
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list) and len(parsed) >= 2:
                return parsed
        except Exception:
            return None
    return None

def prepare_shots(df):
    shots = df.copy()

    # Ensure important columns exist
    required_cols = [
        "team", "player", "minute", "period", "location",
        "shot_outcome", "shot_statsbomb_xg", "shot_body_part",
        "shot_type", "play_pattern"
    ]
    for col in required_cols:
        if col not in shots.columns:
            shots[col] = np.nan

    shots["location_parsed"] = shots["location"].apply(parse_location)
    shots["x"] = shots["location_parsed"].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
    shots["y"] = shots["location_parsed"].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)

    shots["shot_statsbomb_xg"] = pd.to_numeric(shots["shot_statsbomb_xg"], errors="coerce")
    shots["minute"] = pd.to_numeric(shots["minute"], errors="coerce")

    # StatsBomb pitch is 120x80. Goal centre is approximately (120, 40).
    shots["distance_to_goal"] = np.sqrt((120 - shots["x"])**2 + (40 - shots["y"])**2)
    shots["is_goal"] = shots["shot_outcome"].astype(str).str.lower().eq("goal")

    # Clean labels
    for col in ["team", "player", "shot_outcome", "shot_body_part", "shot_type", "play_pattern"]:
        shots[col] = shots[col].fillna("Unknown").astype(str)

    return shots

shots = prepare_shots(shots_raw)
shots[["team", "player", "minute", "x", "y", "shot_outcome", "shot_statsbomb_xg", "distance_to_goal", "is_goal"]].head()


## 4. Dataset overview

Before choosing visualizations, we check the basic structure of the dataset.


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Number of shots",
        "Number of teams",
        "Number of players with shots",
        "Number of goals",
        "Mean xG",
        "Total xG"
    ],
    "Value": [
        len(shots),
        shots["team"].nunique(),
        shots["player"].nunique(),
        int(shots["is_goal"].sum()),
        round(shots["shot_statsbomb_xg"].mean(), 3),
        round(shots["shot_statsbomb_xg"].sum(), 3)
    ]
})

summary


In [ ]:
team_summary = (
    shots
    .groupby("team")
    .agg(
        shots=("team", "size"),
        goals=("is_goal", "sum"),
        total_xg=("shot_statsbomb_xg", "sum"),
        avg_xg=("shot_statsbomb_xg", "mean"),
        avg_distance=("distance_to_goal", "mean")
    )
    .reset_index()
)

team_summary["total_xg"] = team_summary["total_xg"].round(2)
team_summary["avg_xg"] = team_summary["avg_xg"].round(3)
team_summary["avg_distance"] = team_summary["avg_distance"].round(1)

team_summary


## 5. Figure 1 – Shots per team

**Design choice:** A bar chart is appropriate because we compare a small number of categories.
The visual variable is **length**, which is easy to compare accurately.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

plot_data = team_summary.sort_values("shots", ascending=False)
sns.barplot(data=plot_data, x="team", y="shots", ax=ax)

ax.set_title("Figure 1: Number of shots by team")
ax.set_xlabel("Team")
ax.set_ylabel("Number of shots")

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig01_shots_by_team.png", dpi=300, bbox_inches="tight")
plt.show()


**Interpretation:**  
This chart gives the fastest overview of attacking volume.  
The team with more shots had more shooting situations, but this alone does not show shot quality.  
Therefore, we next compare outcomes and expected goals.


## 6. Figure 2 – Shot outcomes by team

**Design choice:** A grouped count plot shows how shot outcomes are distributed between the teams.
This is useful because the same shot volume can still result in very different quality or effectiveness.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

outcome_order = shots["shot_outcome"].value_counts().index.tolist()
sns.countplot(
    data=shots,
    x="shot_outcome",
    hue="team",
    order=outcome_order,
    ax=ax
)

ax.set_title("Figure 2: Shot outcomes by team")
ax.set_xlabel("Shot outcome")
ax.set_ylabel("Number of shots")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig02_shot_outcomes_by_team.png", dpi=300, bbox_inches="tight")
plt.show()


**Interpretation:**  
The outcome plot shows whether the teams mainly produced blocked, saved, off-target or goal-scoring shots.  
This makes the analysis more meaningful than only counting attempts.


## 7. Figure 3 – Total xG by team

**Design choice:** Expected goals (xG) is a quantitative quality indicator.
A bar chart is used again because the task is a direct comparison between teams.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

xg_data = team_summary.sort_values("total_xg", ascending=False)
sns.barplot(data=xg_data, x="team", y="total_xg", ax=ax)

ax.set_title("Figure 3: Total expected goals (xG) by team")
ax.set_xlabel("Team")
ax.set_ylabel("Total xG")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig03_total_xg_by_team.png", dpi=300, bbox_inches="tight")
plt.show()


**Interpretation:**  
xG adds a quality perspective to the analysis.  
A team can have fewer shots but still better chances if those shots are from dangerous positions.


## 8. Figure 4 – Shot map

**Design choice:** Football shots are spatial events.
A pitch map is therefore the most natural representation because it preserves the real-world spatial structure.
Position is encoded by x/y coordinates, colour by team, and marker style by whether the shot was a goal.


In [ ]:
def draw_static_pitch(ax):
    """Draw a simple StatsBomb 120x80 pitch."""
    # Outer lines
    ax.plot([0, 120], [0, 0], color="black")
    ax.plot([0, 120], [80, 80], color="black")
    ax.plot([0, 0], [0, 80], color="black")
    ax.plot([120, 120], [0, 80], color="black")

    # Halfway line and centre circle
    ax.plot([60, 60], [0, 80], color="black")
    centre_circle = plt.Circle((60, 40), 10, color="black", fill=False)
    ax.add_patch(centre_circle)

    # Penalty areas
    ax.plot([0, 18], [18, 18], color="black")
    ax.plot([18, 18], [18, 62], color="black")
    ax.plot([18, 0], [62, 62], color="black")

    ax.plot([120, 102], [18, 18], color="black")
    ax.plot([102, 102], [18, 62], color="black")
    ax.plot([102, 120], [62, 62], color="black")

    # Six-yard boxes
    ax.plot([0, 6], [30, 30], color="black")
    ax.plot([6, 6], [30, 50], color="black")
    ax.plot([6, 0], [50, 50], color="black")

    ax.plot([120, 114], [30, 30], color="black")
    ax.plot([114, 114], [30, 50], color="black")
    ax.plot([114, 120], [50, 50], color="black")

    ax.set_xlim(0, 120)
    ax.set_ylim(80, 0)
    ax.set_aspect("equal")
    ax.axis("off")

fig, ax = plt.subplots(figsize=(10, 6))
draw_static_pitch(ax)

# Non-goals
non_goals = shots[~shots["is_goal"]]
sns.scatterplot(
    data=non_goals,
    x="x", y="y",
    hue="team",
    size="shot_statsbomb_xg",
    sizes=(40, 250),
    alpha=0.75,
    ax=ax
)

# Goals highlighted with star marker
goals = shots[shots["is_goal"]]
ax.scatter(
    goals["x"], goals["y"],
    marker="*",
    s=350,
    edgecolor="black",
    linewidth=1.2,
    label="Goal"
)

ax.set_title("Figure 4: Shot map with goals highlighted")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=4)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig04_shot_map.png", dpi=300, bbox_inches="tight")
plt.show()


**Interpretation:**  
The shot map shows where dangerous moments happened on the pitch.  
Shots close to the centre of the goal usually have a higher chance of scoring.  
By using larger markers for higher xG and star markers for goals, the chart supports both spatial exploration and quick identification of decisive moments.


## 9. Figure 5 – Shot distance distribution

**Design choice:** A histogram is suitable for showing the distribution of shot distances.
This helps compare whether teams mostly shot from close range or from difficult positions.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=shots,
    x="distance_to_goal",
    hue="team",
    bins=8,
    multiple="dodge",
    ax=ax
)

ax.set_title("Figure 5: Distribution of shot distance to goal")
ax.set_xlabel("Distance to goal")
ax.set_ylabel("Number of shots")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig05_shot_distance_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


**Interpretation:**  
This chart adds context to the shot map.  
If one team has many attempts from longer distance, the number of shots can look strong even when the actual chance quality is lower.


## 10. Figure 6 – Timeline of shots

**Design choice:** A timeline is useful because the match is temporal.
Each point represents a shot, the x-position is the match minute, colour shows the team, and marker size represents xG.
Goals are highlighted.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

sns.scatterplot(
    data=shots,
    x="minute",
    y="team",
    size="shot_statsbomb_xg",
    hue="team",
    sizes=(50, 350),
    alpha=0.75,
    ax=ax
)

goal_shots = shots[shots["is_goal"]]
ax.scatter(
    goal_shots["minute"],
    goal_shots["team"],
    marker="*",
    s=400,
    edgecolor="black",
    linewidth=1.2,
    label="Goal"
)

ax.set_title("Figure 6: Shot timeline")
ax.set_xlabel("Match minute")
ax.set_ylabel("Team")
ax.axvline(45, linestyle="--", color="gray", alpha=0.6)
ax.text(45.5, 1.05, "Half-time", color="gray")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig06_shot_timeline.png", dpi=300, bbox_inches="tight")
plt.show()


**Interpretation:**  
The timeline reveals when the main attacking phases happened.  
It also makes it easier to see whether goals followed pressure phases or isolated attacks.


## 11. Figure 7 – Top shooting players

**Design choice:** A horizontal bar chart is suitable for ranking players because names are easier to read on the y-axis.


In [ ]:
player_summary = (
    shots
    .groupby(["player", "team"])
    .agg(
        shots=("player", "size"),
        goals=("is_goal", "sum"),
        total_xg=("shot_statsbomb_xg", "sum")
    )
    .reset_index()
    .sort_values(["shots", "total_xg"], ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=player_summary,
    x="shots",
    y="player",
    hue="team",
    dodge=False,
    ax=ax
)

ax.set_title("Figure 7: Top shooting players")
ax.set_xlabel("Number of shots")
ax.set_ylabel("Player")

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig07_top_shooting_players.png", dpi=300, bbox_inches="tight")
plt.show()

player_summary


**Interpretation:**  
The player ranking identifies who contributed most to the attacking output.
This is useful for storytelling because it connects abstract shot events to concrete players.


## 12. Design reflection for GDV

The static visualizations follow basic principles from data visualization theory:

- **Position and length** are used for precise comparisons, for example in bar charts.
- **Spatial position** is used for football shot locations because pitch coordinates are meaningful.
- **Colour** is used only for categorical separation between teams.
- **Marker size** is used carefully for xG because size is less precise than position, but still helpful for rough salience.
- **Goal markers** use a different shape to create a pre-attentive highlight.
- The notebook avoids unnecessary decoration and keeps axes, labels and titles explicit.

These decisions support the GDV goal of connecting data, design and interpretation.


## 13. Main insights for report and pitch

The following insights can be used later in the written report and presentation:

1. Shot count alone is not enough; xG and distance give important quality context.
2. The shot map shows where the dangerous moments happened spatially.
3. The timeline shows when attacking pressure and goals happened during the match.
4. Player-level analysis helps connect team performance to individual contributors.
5. The static GDV analysis provides the basis for the later interactive IVI dashboard.

The figures saved in `reports/figures/` can be used in the final report and pitch slides.
